In [2]:
# Local environment check for QuimicaAutomocio P2.
# The original eChem notebooks are Colab-oriented; this one is designed for local use.
import numpy as np
import py3Dmol
import veloxchem as vlx

required = ["ScfRestrictedDriver", "OptimizationDriver", "VibrationalAnalysis"]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")

VeloxChem 1.0rc4
Environment OK


In [3]:
def read_pdb_coordinates(pdb_path, chain=None, residue_name=None):
    """Parse a PDB file and return atom symbols + coordinates."""
    atoms = []
    coords = []

    with open(pdb_path, "r") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                rec_chain = line[21].strip()
                rec_resname = line[17:20].strip()

                if chain is not None and rec_chain != chain:
                    continue
                if residue_name is not None and rec_resname != residue_name:
                    continue

                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                element = line[76:78].strip()
                if not element:
                    atom_name = line[12:16].strip()
                    element = "".join([c for c in atom_name if c.isalpha()])[:2]
                    element = element.capitalize()

                atoms.append(element)
                coords.append((x, y, z))

    return atoms, np.array(coords)


def to_xyz_string(atoms, coords, comment="Structure from PDB"):
    lines = [str(len(atoms)), comment]
    for atom, (x, y, z) in zip(atoms, coords):
        lines.append(f"{atom:2s} {x:12.6f} {y:12.6f} {z:12.6f}")
    return "\n".join(lines)


# --- Files ---
pdb_files = {
    "RS": "step_1_1_RS.pdb",
    "TS": "step_1_1_TS.pdb",
    "PS": "step_1_1_PS.pdb",
}

# If each PDB has the whole protein+solvent, set this to the ligand's
# residue name (e.g. "LIG") to extract just the reactive fragment.
# Leave as None to read the entire system.
residue_filter = None

molecules = {}
for label, path in pdb_files.items():
    atoms, coords = read_pdb_coordinates(path, residue_name=residue_filter)
    xyz_string = to_xyz_string(atoms, coords, comment=f"{label} structure")
    molecules[label] = vlx.Molecule.read_xyz_string(xyz_string)
    print(f"{label}: {len(atoms)} atoms read from {path}")

# Access each one:
molecule_rs = molecules["RS"]
molecule_ts = molecules["TS"]
molecule_ps = molecules["PS"]

print(molecule_rs.get_xyz_string())
molecule_rs.show(atom_indices=True, width=520, height=360)

RS: 57 atoms read from step_1_1_RS.pdb
TS: 57 atoms read from step_1_1_TS.pdb
PS: 57 atoms read from step_1_1_PS.pdb
57

N              0.000000000000         0.000000000000         0.000000000000
C              1.460000000000         0.000000000000         0.000000000000
C              2.155000000000         0.774000000000         0.922000000000
C              3.545000000000         0.774000000000         0.922000000000
C              4.240000000000        -0.000000000000        -0.000000000000
C              3.545000000000        -0.774000000000        -0.922000000000
C              2.155000000000        -0.774000000000        -0.922000000000
C              5.740000000000        -0.000000000000        -0.000000000000
C              6.240000000000         1.083000000000        -0.909000000000
C              0.967000000000         0.010000000000        -2.840000000000
C              1.720000000000         1.308000000000        -2.840000000000
C              1.073000000000         2.482

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.dispersion = True
scf_drv.ostream.mute()

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.ostream.mute()
opt_drv.max_iter = 100

opt_results = opt_drv.compute(molecule, basis)
opt_molecule = opt_results["final_molecule"]

print("Optimized geometry:")
print(opt_molecule.get_xyz_string())
print("Optimization steps:", len(opt_results.get("opt_energies", [])))
opt_molecule.show(atom_indices=True, width=520, height=360)

NameError: name 'molecule' is not defined

In [ ]:
vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_drv.ostream.mute()
vib_drv.do_ir = True

vib_results = vib_drv.compute(opt_molecule, basis)

frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
normal_modes = np.array(vib_results["normal_modes"], dtype=float)
reduced_masses = np.array(vib_results["reduced_masses"], dtype=float)
force_constants = np.array(vib_results["force_constants"], dtype=float)
ir_intensities = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)

print("mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1")
for i, freq in enumerate(frequencies, start=1):
    print(f"{i:>4d}  {freq:>16.2f}  {reduced_masses[i-1]:>15.4f}  {force_constants[i-1]:>25.4f}  {ir_intensities[i-1]:>14.2f}")

mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1
   1             20.87           4.0608                     0.0010            0.47
   2             26.92           3.8220                     0.0016            0.02
   3             52.18           2.7884                     0.0045            0.19
   4             80.78           2.5432                     0.0098            1.24
   5             92.77           1.6105                     0.0082            0.17
   6            100.89           1.9535                     0.0117            0.47
   7            124.92           2.2245                     0.0205            0.44
   8            135.48           2.8889                     0.0312            3.06
   9            169.05           3.9856                     0.0671            0.81
  10            194.63           2.0775                     0.0464            0.27
  11            220.04           2.3196                     0.0662            3.30
 

In [ ]:
def normal_mode_trajectory(molecule, mode, amplitude=0.45, frames=32):
    labels = list(molecule.get_labels())
    coords = np.array(molecule.get_coordinates_in_angstrom(), dtype=float)
    mode = np.array(mode, dtype=float)

    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(molecule, mode_index, amplitude=0.45, frames=32):
    traj = normal_mode_trajectory(molecule, normal_modes[mode_index], amplitude=amplitude, frames=frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Python index: 0 is the first vibrational mode. Change this value.
mode_index = 0
print(f"Showing mode {mode_index}: {frequencies[mode_index]:.2f} cm^-1")
show_mode(opt_molecule, mode_index, amplitude=0.45, frames=32).show()
print(normal_modes[mode_index])

Showing mode 0: 20.87 cm^-1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

[[-0.006207    0.0378014  -0.22632298]
 [ 0.04811735  0.15439264 -0.10545699]
 [-0.02963615  0.10265559  0.03375235]
 [-0.05657672  0.03919601  0.03943644]
 [-0.06645555  0.0204846   0.04360852]
 [ 0.01016729 -0.12029857 -0.03768798]
 [ 0.23428876 -0.13726822 -0.10931501]
 [-0.03225456  0.00589442  0.03975276]
 [-0.01228424 -0.01516103  0.04306749]
 [-0.00734126 -0.02584616  0.04683494]
 [ 0.00402119 -0.03040782  0.04508934]
 [-0.00757511 -0.01925896  0.04350553]
 [-0.00106245 -0.02662442  0.04432263]
 [-0.01909875 -0.0089406   0.04279636]
 [-0.04310226  0.01532123  0.04017491]
 [-0.04909455  0.02192102  0.03938562]
 [-0.03104808  0.00423715  0.04058715]
 [-0.07041602 -0.02372444 -0.15548344]
 [ 0.05005027  0.08213504 -0.31899185]
 [-0.04132328 -0.03546241 -0.29310097]
 [ 0.08858864  0.23098833 -0.04225023]
 [ 0.11712196  0.21585964 -0.18041287]
 [-0.0323819   0.19424922  0.11882325]
 [-0.07205842  0.0275975   0.06173462]
 [-0.08961473  0.04233076  0.04662876]
 [-0.02856997  0.12584305